# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment (Python 3.8 or newer).

It uses [Skyfield](https://rhodesmill.org/skyfield/), a standard Python library for satellite positions, and the helper file `Satellites/sat_tools.py` next to this notebook. The orbit data of all satellites is in `Satellites/tle/`: a frozen snapshot from [CelesTrak](https://celestrak.org/), so the notebook needs no internet after the install. Keep the folder next to the notebook.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install skyfield numpy matplotlib --quiet

**Imports**

In [ ]:
# --- Imports: libraries and functions used in this notebook ---
import numpy as np
import matplotlib.pyplot as plt

from Satellites.sat_tools import (SNAPSHOT_DAY, GNSS, load_group, constellations, orbit_basics, swiss_time_to_utc,
                                  gnss_sky, plot_sky, sky_table, street_sweep, customer_for, serve_customer,
                                  walker_shell, evaluate_shell, beam_capacity, DISH_BEAM, PHONE_BEAM,
                                  constellation_average, masts_needed, bridge_figure, next_passes)

### 1. Real Satellites, Frozen in Time

Every satellite's orbit is published as a **two-line element set (TLE)**: six numbers that describe an ellipse (size, shape, tilt, orientation, position on it) plus a drag term, valid around one moment, the epoch. The **SGP4** model turns a TLE into a position at any time. It is accurate to about a kilometre near the epoch and slowly degrades over weeks, which is accurate enough to know where a satellite is in your sky.

The snapshot in `Satellites/tle/` holds all active satellites of these systems:

| System | Use | Orbit |
|---|---|---|
| GPS, Galileo, GLONASS, BeiDou | navigation (GNSS) | medium Earth orbit, about 19,000 to 23,000 km, 55° to 65° tilt (BeiDou also uses geostationary and inclined geosynchronous satellites) |
| Starlink | broadband to dishes, and to phones | low Earth orbit, about 470 km, in shells tilted 43°, 53°, 70° and 97.6° |
| OneWeb | broadband to dishes | about 1,200 km, 88° tilt |
| Iridium | voice and data to handhelds | about 780 km, 86° tilt |
| geostationary belt | TV, broadband, maritime | 35,786 km above the equator |

**Declarations**

Your customer in section 3 depends on your matriculation number.

In [ ]:
# --- Declarations: editable variables for students ---
MATRICULATION = "12-345-678"     # <-- Change this. Your matriculation number

print(f"Orbit snapshot: {SNAPSHOT_DAY:%Y-%m-%d}")
for system, group in GNSS.items():
    print(f"   {system:10s}{len(load_group(group)):6d} satellites")
for label, s in constellations().items():
    print(f"   {label:15s}{len(s['satrecs']):6d} satellites ({s['note']})")

**Orbits in numbers.** A circular orbit's period follows from Kepler's third law, $T = 2\pi\sqrt{a^3/\mu}$, with $a$ the distance from the Earth's centre and $\mu = 398{,}600\ \text{km}^3/\text{s}^2$. The higher the orbit, the slower the satellite and the larger the area that sees it (its **footprint**, here above 25° elevation). At 35,786 km the period equals one day: the satellite stays above one point of the equator.

In [ ]:
for name, h in (("Starlink", 470), ("OneWeb", 1200), ("GPS", 20200), ("geostationary", 35786)):
    o = orbit_basics(h)
    print(f"{name:14s}{h:7,d} km: period {o['period_min']:7.1f} min, speed {o['speed_kms']:4.1f} km/s, "
          f"footprint radius at 25° {o['footprint_radius_km']:6,.0f} km, delay straight down {h / 299.792:6.1f} ms")

### 2. The GNSS Sky Over You

A GNSS receiver needs at least four satellites: three for its position and one more for its clock error. How well it can fix its position depends on how many satellites it sees and on their **geometry**: satellites spread over the whole sky give a sharp fix, satellites bunched in one direction a blurred one. The **dilution of precision (DOP)** measures this: the position error is about the DOP times the error of each range measurement (about 3 m for a phone in the open). PDOP is for the 3D position, HDOP for the horizontal one.

**Street canyons.** Between buildings, a phone only sees the strip of sky above the street. For a straight street of width $w$ with walls of height $h$, a satellite at azimuth $\alpha$ (relative to the street's axis) is hidden below the elevation $\arctan\left(h / \tfrac{w/2}{|\sin\alpha|}\right)$: along the street the sky is open, across it the walls hide everything below $\arctan(2h/w)$.

Enter the place and time of your measurement (Swiss local time; on map.geo.admin.ch or any map, a right-click shows the coordinates of a point), and the street's width, wall height and direction.

In [ ]:
# --- Your measurement ---
LAT, LON  = 47.3763, 8.5480                  # <-- Change this. Where you measured (degrees)
WHEN      = "2026-11-12 14:05"               # <-- Change this. Swiss local time of the measurement
STREET    = dict(width_m=12, height_m=15,    # <-- Change this. Street width and wall height (m)
                 direction_deg=0)            # <-- Change this. Direction of the street (0 = north-south, 90 = east-west)

when_utc = swiss_time_to_utc(WHEN)
sky = gnss_sky(LAT, LON, when_utc)
plot_sky(sky, STREET, title=f"GNSS sky at {LAT:.4f}, {LON:.4f}, {WHEN} (Swiss time)")
rows = sky_table(sky, STREET)

#### 2.1 A Street Canyon Over a Whole Day

The satellites move, so one moment can be lucky or unlucky. Over a whole day (the snapshot day, every 10 minutes): in how much of the time does a phone in a street still see at least four satellites, and how good is the geometry?

In [ ]:
sweep = street_sweep(LAT, LON, height_m=STREET["height_m"], direction_deg=STREET["direction_deg"])

### 3. Your Customer

You now plan for a customer far from any mast. Each customer has a place, a need and a terrain mask (mountains or valley walls that hide the low sky). Your job: which real system can serve them, how well, and why.

**What decides coverage**
- **Inclination.** A satellite never flies over latitudes higher than its orbit's tilt, and it spends most of its time near that latitude. A shell serves latitudes up to about its inclination plus its footprint (a few degrees in low orbit). Polar shells pass over every place.
- **Elevation.** Dishes and phones need the satellite above a minimum elevation (25° for broadband dishes here, 8° for Iridium, 10° for a geostationary dish), and above the terrain.
- **Geostationary satellites** sit over the equator. From high latitudes they appear low on the horizon; beyond about 81° they are below it.

**What the link sees**
- **Delay**: the distance divided by the speed of light. An internet round trip goes to the satellite and down to a gateway, and back: about four times the one-way delay in the table.
- **Doppler**: a low satellite moves at 7.6 km/s, so its carrier shifts by up to about ±40 kHz at 2 GHz while it passes. An LTE or 5G subcarrier is only 15 kHz wide, so phones that talk to satellites directly (3GPP "non-terrestrial networks") must pre-correct their frequency and timing, from their own GNSS position and the satellite's orbit.

The table and plot cover one day at 1-minute steps.

In [ ]:
customer = customer_for(MATRICULATION)
result = serve_customer(customer)

### 4. A What-If Shell

Could you serve your customer with fewer satellites? A **Walker constellation** places `planes` orbital planes evenly around the Earth, each with `per_plane` satellites, all at the same altitude and inclination. Things to try:
- **Inclination close to your customer's latitude** puts the satellites where they linger longest. Too low, and they never come close enough.
- **Higher altitude** gives a larger footprint, so fewer satellites cover the same area, but the delay grows and the link gets weaker.
- **Fewer planes** leave longer gaps between passes; fewer satellites per plane leave gaps along each plane.

Each run takes a few seconds.

In [ ]:
SHELL = dict(alt_km=550, inc_deg=53, planes=24, per_plane=20)     # <-- Change this. Your shell
MIN_EL = 25                                                        # <-- Change this. Minimum elevation of the terminal

shell = walker_shell(SHELL["alt_km"], SHELL["inc_deg"], SHELL["planes"], SHELL["per_plane"])
mine = evaluate_shell(customer, shell, min_el=MIN_EL, label=f"{len(shell)} sats, {SHELL['inc_deg']}°")

### 5. Masts or Satellites?

**Capacity from space.** A satellite serves many **spot beams**, and the beams reuse the frequencies like cells (a reuse factor of about 4 keeps neighbouring beams apart). A beam's width is about $1.02\,\lambda/D$ for an antenna of diameter $D$, so its area on the ground is fixed by the antenna and the altitude, and so is its capacity per km²:

- a **dish service** (Ku band, 11.7 GHz, a 1 m satellite antenna, 2 GHz of spectrum shared by 4 beam colours, 3 b/s/Hz);
- a **direct-to-phone service** (1.9 GHz, a 5 m satellite antenna, 5 MHz of spectrum, 1 b/s/Hz: a phone's small antenna allows little more).

A constellation adds capacity by adding satellites: about 1,500 to 5,000 satellites already cover the latitudes where people live, but each carries only tens of Gb/s, spread over a huge footprint. So **coverage takes hundreds of satellites, capacity takes thousands**, just as a city needs far more masts for capacity than for coverage (section 6 of `Network_Cellular.ipynb`).

**Masts on the ground.** A 30 m mast reaches at most its radio horizon, about 28 km for a phone at 1.5 m; in hills much less. Below the cellular knee, masts are mostly idle: they are paid for coverage, not capacity.

The bridge figure puts both on one picture, with the cellular model of `Network_Cellular.ipynb`. Add your own site estimate from the tram ride there (`MY_SITES_PER_KM2`).

In [ ]:
MY_SITES_PER_KM2 = None          # <-- Change this. Your tram-ride estimate from Network_Cellular.ipynb, section 7

for name, b in (("dish beam", DISH_BEAM), ("direct-to-phone beam", PHONE_BEAM)):
    r = beam_capacity(**b)
    print(f"{name:22s}: {r['beam_area_km2']:6.0f} km2, {r['beam_mbps']:7.1f} Mb/s per beam, {r['mbps_per_km2']:.3f} Mb/s per km2")
m = masts_needed(customer)
print(f"masts of 30 m for your customer: reach at most {m['reach_km']:.0f} km each -> at least {m['masts']} mast(s)")
print(f"   {m['note']}\n" if m["note"] else "")
bridge = bridge_figure(customer, my_sites_per_km2=MY_SITES_PER_KM2)

### 6. Optional: Look Up

The International Space Station passes over Switzerland several times a day. Around dawn and dusk it shines brightly in the sunlight while the sky is dark. Passes near the snapshot date (other dates: the orbit drifts, so check a site such as heavens-above.com):

In [ ]:
next_passes(LAT, LON, "ISS (ZARYA)", days=3)